In [143]:
# CELL 1: Libraries & Data Extraction
import pandas as pd
import subprocess
import os
import xml.etree.ElementTree as ET

# --- 1. Parse the Network File for Street Names ---
net_file = 'network/osm_no_uturns.net.xml'
tree = ET.parse(net_file)
root = tree.getroot()

edge_list = []
for edge in root.findall('edge'):
    eid = edge.get('id')
    if not eid or eid.startswith(':'): continue
    
    street_name = "Unknown"
    # Check attributes and nested parameters for the name
    if edge.get('name'):
        street_name = edge.get('name')
    else:
        for param in edge.findall('param'):
            if param.get('key') in ['osm_name', 'name']:
                street_name = param.get('value')
    
    edge_list.append({'edge_id': eid, 'street_name': street_name})

network_df = pd.DataFrame(edge_list)
print(f"✅ Map Processed: Found {len(network_df)} edges.")

✅ Map Processed: Found 5243 edges.


In [145]:
# CELL 2: Automated Calibration Generator
# 1. Map Station IDs to names
search_map = {
    '#520099000000': 'East Main Street',
    '#520020000000': 'Bennett Road', 
    '#520098000000': 'West Main Street',
    '#526301000000': 'Temple Road'
}

# 2. Load Data
dot_df = pd.read_csv('aadt_and_truckpct.csv', low_memory=False)

xml_output = '<additional>\n'
xml_output += '    <vType id="passenger_car" vClass="passenger" length="5.0" sigma="0.5"/>\n'
xml_output += '    <vType id="heavy_truck" vClass="truck" length="12.0" sigma="0.5"/>\n\n'

for station_id, street_name in search_map.items():
    matches = network_df[network_df['street_name'] == street_name]
    
    if not matches.empty:
        entry_edge = matches.sort_values('edge_id').iloc[0]['edge_id']
        row = dot_df[dot_df['Station ID'] == station_id].iloc[0]
        
        total_flow = (row['AADT_2024'] * 0.10) / 2
        truck_pct = row['Truck%_2024'] if pd.notna(row['Truck%_2024']) else 5.0
        
        car_vol = int(total_flow * (1 - truck_pct/100))
        truck_vol = int(total_flow * (truck_pct/100))
        
        # THE BULLETPROOF VERSION: Two distinct intervals within the calibrator
        xml_output += f'    \n'
        xml_output += f'    <calibrator id="cal_{entry_edge}" edge="{entry_edge}" pos="0">\n'
        xml_output += f'        <interval begin="0" end="3600" vehsPerHour="{car_vol}" type="passenger_car"/>\n'
        xml_output += f'        <interval begin="0" end="3600" vehsPerHour="{truck_vol}" type="heavy_truck"/>\n'
        xml_output += f'    </calibrator>\n\n'
    else:
        print(f"⚠️ Warning: {street_name} not found in map!")

xml_output += '</additional>'

with open('network/calibrators.add.xml', 'w') as f:
    f.write(xml_output)

print("✅ Calibrators regenerated using explicit interval attributes.")

✅ Calibrators regenerated using explicit interval attributes.


In [151]:
# CELL 3: The Simulation Engine
def run_scenario(name, extra_routes=None):
    if not os.path.exists('results'): 
        os.makedirs('results')
        
    print(f"🚀 Running Scenario: {name}...")
    
    # Ensure a basic output.add.xml exists to prevent crashing
    if not os.path.exists('network/output.add.xml'):
        with open('network/output.add.xml', 'w') as f: 
            f.write('<additional></additional>')

    # Determine the route string
    route_files = "demand/osm.passenger.trips.xml"
    if extra_routes:
        route_files += f",{extra_routes}"

    cmd = [
        "sumo-gui", 
        "-n", "network/osm_no_uturns.net.xml",
        "-r", route_files,
        "-a", "network/calibrators.add.xml,network/output.add.xml",
        "--tripinfo-output", f"results/tripinfo_{name}.xml",
        "--start", 
        #"--quit-on-end",
        "--end", "3600"  # Ensures the simulation runs for 1 hour
    ]
    
    subprocess.run(cmd)
    print(f"✅ {name} complete. Data saved to results/tripinfo_{name}.xml")

In [153]:
#CELL 4: Running Scenarios
# --- 1. Run the Baseline ---
run_scenario("Baseline")

# --- 2. Run the Bus Scenario ---
# Note: Ensure demand/buses.rou.xml actually exists before running this!
if os.path.exists("demand/buses.rou.xml"):
    run_scenario("With_Buses", extra_routes="demand/buses.rou.xml")
else:
    print("⚠️ Skipping 'With_Buses' because demand/buses.rou.xml was not found.")

🚀 Running Scenario: Baseline...
✅ Baseline complete. Data saved to results/tripinfo_Baseline.xml
🚀 Running Scenario: With_Buses...
✅ With_Buses complete. Data saved to results/tripinfo_With_Buses.xml


In [155]:
#CELL 5: Data Analysis
def get_delay(name):
    path = f"results/tripinfo_{name}.xml"
    if not os.path.exists(path):
        return 0
    tree = ET.parse(path)
    root = tree.getroot()
    delays = [float(t.get('waitingTime')) for t in root.findall('tripinfo')]
    return sum(delays) / len(delays) if delays else 0

base = get_delay("Baseline")
bus = get_delay("With_Buses")

print(f"\n--- FREDONIA TRAFFIC STUDY ---")
print(f"Avg Delay Baseline: {base:.2f} seconds")
print(f"Avg Delay w/ Buses: {bus:.2f} seconds")

if base > 0:
    impact = ((bus - base) / base) * 100
    print(f"Result: {impact:.1f}% change in traffic delay.")
else:
    print("Result: Baseline delay is still 0. Check if cars are appearing in the GUI!")


--- FREDONIA TRAFFIC STUDY ---
Avg Delay Baseline: 185.17 seconds
Avg Delay w/ Buses: 0.00 seconds
Result: -100.0% change in traffic delay.
